# Task 1C: Feature Engineering

Transform the mental health smartphone time series dataset into an instance-based dataset suitable for supervised learning.

**Goal:** Predict next-day average mood from historical features using a sliding window approach.

**Steps:**
1. Load cleaned data (with raw fallback)
2. Aggregate raw observations into daily summaries per patient
3. Apply sliding window feature engineering (mean, std, min, max, trend, lags)
4. Build final instance-based dataset: each row = (patient, date, features, target)
5. Analyze feature quality and correlations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("Libraries loaded successfully.")

## Section 1: Load Cleaned Data

We try to load the cleaned dataset from Task 1B. If that file is not available, we fall back to loading and cleaning the raw data directly.

In [ ]:
import os

cleaned_path = '../data/dataset_cleaned.csv'
raw_path = '../data/dataset_mood_smartphone.csv'

if os.path.exists(cleaned_path):
    df = pd.read_csv(cleaned_path)
    print(f"Loaded cleaned dataset from {cleaned_path}")
else:
    print(f"Cleaned dataset not found at {cleaned_path}. Loading raw data as fallback...")
    df = pd.read_csv(raw_path)
    # Drop the unnamed index column if present
    if df.columns[0] == '' or df.columns[0].startswith('Unnamed'):
        df = df.drop(columns=df.columns[0])
    print(f"Loaded raw dataset from {raw_path}")

# Ensure correct types
df['time'] = pd.to_datetime(df['time'])
df['date'] = df['time'].dt.date
df['value'] = pd.to_numeric(df['value'], errors='coerce')

print(f"\nDataset shape: {df.shape}")
print(f"Number of patients: {df['id'].nunique()}")
print(f"Variables: {sorted(df['variable'].unique())}")
print(f"Date range: {df['time'].min()} to {df['time'].max()}")
df.head()

## Section 2: Daily Aggregation

Convert the long-format time series (multiple observations per day) into wide-format daily summaries per patient. Each variable is aggregated differently based on its nature:

- **mood**: daily mean (self-reported scale 1-10)
- **circumplex.arousal / circumplex.valence**: daily mean
- **activity**: daily mean
- **screen**: daily sum (total screen time)
- **call**: daily count (number of calls)
- **sms**: daily count (number of messages)
- **appCat.\***: daily sum (total usage per category)

In [ ]:
# Define aggregation strategy per variable type
mean_vars = ['mood', 'circumplex.arousal', 'circumplex.valence', 'activity']
sum_vars = ['screen'] + [v for v in df['variable'].unique() if v.startswith('appCat.')]
count_vars = ['call', 'sms']

print("Variables aggregated by MEAN:", mean_vars)
print("Variables aggregated by SUM:", sum_vars)
print("Variables aggregated by COUNT:", count_vars)

In [ ]:
def aggregate_daily(df, mean_vars, sum_vars, count_vars):
    """
    Aggregate raw observations into one row per patient per day.
    Returns a wide-format DataFrame with columns: id, date, <variable_1>, <variable_2>, ...
    """
    daily_frames = []

    for var in mean_vars:
        sub = df[df['variable'] == var].groupby(['id', 'date'])['value'].mean().reset_index()
        sub = sub.rename(columns={'value': var})
        daily_frames.append(sub)

    for var in sum_vars:
        sub = df[df['variable'] == var].groupby(['id', 'date'])['value'].sum().reset_index()
        sub = sub.rename(columns={'value': var})
        daily_frames.append(sub)

    for var in count_vars:
        sub = df[df['variable'] == var].groupby(['id', 'date'])['value'].count().reset_index()
        sub = sub.rename(columns={'value': var})
        daily_frames.append(sub)

    # Merge all variable aggregations on (id, date)
    daily = daily_frames[0]
    for frame in daily_frames[1:]:
        daily = daily.merge(frame, on=['id', 'date'], how='outer')

    daily = daily.sort_values(['id', 'date']).reset_index(drop=True)
    return daily

daily_df = aggregate_daily(df, mean_vars, sum_vars, count_vars)

print(f"Daily aggregated shape: {daily_df.shape}")
print(f"Patients: {daily_df['id'].nunique()}, Days per patient (approx): {daily_df.groupby('id')['date'].count().mean():.0f}")
print(f"\nColumns: {list(daily_df.columns)}")
daily_df.head(10)

In [ ]:
# Check missing values in the daily aggregated data
missing = daily_df.isnull().sum()
missing_pct = (daily_df.isnull().sum() / len(daily_df) * 100).round(1)
missing_info = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print("Missing values in daily aggregated data:")
print(missing_info[missing_info['missing_count'] > 0])

# Fill missing daily values with 0 for count/sum variables (no observation = no activity)
fill_zero_cols = [c for c in sum_vars + count_vars if c in daily_df.columns]
daily_df[fill_zero_cols] = daily_df[fill_zero_cols].fillna(0)

print(f"\nAfter filling count/sum NaNs with 0: {daily_df.isnull().sum().sum()} remaining NaNs")
print("Remaining NaN columns:", list(daily_df.columns[daily_df.isnull().any()]))

## Section 3: Sliding Window Feature Engineering

For each patient, we slide a window of `W` days over their daily time series. From each window we extract:

- **Aggregation features**: mean, std, min, max of each variable over the window
- **Trend features**: linear slope of each variable over the window (captures direction of change)
- **Lag features**: mood values from the previous 1, 2, and 3 days
- **Temporal features**: day of the week (cyclical encoding)
- **Target**: the average mood on the *next* day after the window

In [ ]:
# Configuration
WINDOW_SIZE = 7  # number of days in the history window
N_LAGS = 3       # number of mood lag features

# Feature columns (all variables except id and date)
feature_vars = [c for c in daily_df.columns if c not in ['id', 'date']]
print(f"Window size: {WINDOW_SIZE} days")
print(f"Number of lag features: {N_LAGS}")
print(f"Feature variables: {feature_vars}")

In [ ]:
def compute_slope(series):
    """Compute linear trend (slope) over a series. Returns NaN if not enough data."""
    y = series.dropna()
    if len(y) < 2:
        return np.nan
    x = np.arange(len(y))
    slope, _, _, _, _ = stats.linregress(x, y)
    return slope


def build_sliding_window_features(patient_df, feature_vars, window_size, n_lags):
    """
    Build feature rows from a single patient's daily data using a sliding window.
    
    For each valid position t (where we have window_size days of history and
    a next-day target), we create one instance with:
    - Rolling aggregation features (mean, std, min, max) over the window
    - Trend (slope) over the window
    - Lag features for mood
    - Day-of-week (sin/cos encoded)
    - Target: next-day mood
    """
    patient_df = patient_df.sort_values('date').reset_index(drop=True)
    n_days = len(patient_df)
    instances = []

    for t in range(window_size, n_days):
        # Window: days [t - window_size, t) ; Target: day t
        window = patient_df.iloc[t - window_size : t]
        target_row = patient_df.iloc[t]

        # Skip if target mood is missing
        if pd.isna(target_row['mood']):
            continue

        row = {
            'id': patient_df.iloc[t]['id'],
            'date': target_row['date'],
            'target_mood': target_row['mood'],
        }

        # Aggregation features over the window
        for var in feature_vars:
            vals = window[var]
            row[f'{var}_mean'] = vals.mean()
            row[f'{var}_std'] = vals.std()
            row[f'{var}_min'] = vals.min()
            row[f'{var}_max'] = vals.max()
            row[f'{var}_trend'] = compute_slope(vals)

        # Lag features for mood (mood on day t-1, t-2, t-3, ...)
        for lag in range(1, n_lags + 1):
            lag_idx = t - lag
            if lag_idx >= 0:
                row[f'mood_lag{lag}'] = patient_df.iloc[lag_idx]['mood']
            else:
                row[f'mood_lag{lag}'] = np.nan

        # Day-of-week features (cyclical encoding)
        target_date = pd.to_datetime(target_row['date'])
        dow = target_date.dayofweek  # 0=Monday, 6=Sunday
        row['dow_sin'] = np.sin(2 * np.pi * dow / 7)
        row['dow_cos'] = np.cos(2 * np.pi * dow / 7)

        instances.append(row)

    return pd.DataFrame(instances)


print("Feature engineering function defined.")

In [ ]:
# Apply sliding window feature engineering per patient
patient_ids = sorted(daily_df['id'].unique())
all_instances = []

for pid in patient_ids:
    patient_data = daily_df[daily_df['id'] == pid].copy()
    patient_features = build_sliding_window_features(
        patient_data, feature_vars, WINDOW_SIZE, N_LAGS
    )
    all_instances.append(patient_features)

features_df = pd.concat(all_instances, ignore_index=True)

print(f"Total instances created: {len(features_df)}")
print(f"Patients represented: {features_df['id'].nunique()}")
print(f"Instances per patient:")
print(features_df.groupby('id').size().describe())
print(f"\nTotal features: {len([c for c in features_df.columns if c not in ['id', 'date', 'target_mood']])})")

In [ ]:
# Preview the feature matrix
print("Feature columns:")
feature_cols = [c for c in features_df.columns if c not in ['id', 'date', 'target_mood']]
for i, col in enumerate(feature_cols):
    print(f"  {i+1:3d}. {col}")

print(f"\nTarget distribution:")
print(features_df['target_mood'].describe())
features_df.head()

## Section 4: Create Final Dataset

Combine all features into the final instance-based dataset and save to disk. Each row represents one training instance with the structure:

`(patient_id, date, feature_1, feature_2, ..., feature_N, target_mood)`

In [ ]:
# Handle remaining NaN values
print("NaN counts before handling:")
nan_counts = features_df.isnull().sum()
print(nan_counts[nan_counts > 0])

# For features with NaN (e.g., from std of constant windows, or missing arousal/valence),
# fill with 0 for std columns and forward-fill or median for others
std_cols = [c for c in features_df.columns if c.endswith('_std')]
features_df[std_cols] = features_df[std_cols].fillna(0)

# For remaining NaN features, fill with column median per patient
remaining_nan_cols = features_df.columns[features_df.isnull().any()]
for col in remaining_nan_cols:
    if col in ['id', 'date', 'target_mood']:
        continue
    features_df[col] = features_df.groupby('id')[col].transform(
        lambda x: x.fillna(x.median())
    )
    # If still NaN (entire patient missing), fill with global median
    features_df[col] = features_df[col].fillna(features_df[col].median())

# Drop any remaining rows where target is missing
features_df = features_df.dropna(subset=['target_mood']).reset_index(drop=True)

print(f"\nFinal dataset shape: {features_df.shape}")
print(f"Remaining NaN values: {features_df.isnull().sum().sum()}")

In [ ]:
# Save the final feature-engineered dataset
output_path = '../data/dataset_features.csv'
features_df.to_csv(output_path, index=False)
print(f"Saved feature-engineered dataset to {output_path}")
print(f"Shape: {features_df.shape}")
print(f"  - Rows (instances): {len(features_df)}")
print(f"  - Columns: {len(features_df.columns)} (id + date + {len(feature_cols)} features + target_mood)")

## Section 5: Feature Analysis

Examine the relationship between engineered features and the target variable (next-day mood). This helps identify which features are most informative and guides feature selection for modelling.

In [ ]:
# Compute correlations between all features and the target
feature_cols = [c for c in features_df.columns if c not in ['id', 'date', 'target_mood']]
correlations = features_df[feature_cols].corrwith(features_df['target_mood']).sort_values(ascending=False)

# Show top 20 positively and negatively correlated features
print("Top 20 features most POSITIVELY correlated with next-day mood:")
print(correlations.head(20).to_string())
print(f"\nTop 20 features most NEGATIVELY correlated with next-day mood:")
print(correlations.tail(20).to_string())

In [ ]:
# Visualize top correlated features
top_n = 30
top_corr = pd.concat([correlations.head(top_n // 2), correlations.tail(top_n // 2)])

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in top_corr.values]
top_corr.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Pearson Correlation with Next-Day Mood')
ax.set_title(f'Top {top_n} Features by Correlation with Target')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance using a quick Random Forest
from sklearn.ensemble import RandomForestRegressor

X = features_df[feature_cols].copy()
y = features_df['target_mood'].copy()

# Drop any columns that are all NaN or constant
valid_cols = X.columns[X.std() > 0]
X = X[valid_cols]

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

print("Top 20 features by Random Forest importance:")
print(importances.head(20).to_string())
print(f"\nRF R-squared (training, for reference): {rf.score(X, y):.3f}")

In [ ]:
# Visualize Random Forest feature importances
fig, ax = plt.subplots(figsize=(10, 10))
importances.head(30).sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Feature Importance (MDI)')
ax.set_title('Top 30 Features by Random Forest Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Target distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(features_df['target_mood'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Next-Day Average Mood')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Target Variable (Next-Day Mood)')

# Scatter: mood_lag1 vs target
axes[1].scatter(features_df['mood_lag1'], features_df['target_mood'], alpha=0.3, s=10)
axes[1].set_xlabel('Previous Day Mood (lag 1)')
axes[1].set_ylabel('Next-Day Mood (target)')
axes[1].set_title('Mood Autocorrelation: Lag-1 vs Target')
r = features_df[['mood_lag1', 'target_mood']].corr().iloc[0, 1]
axes[1].annotate(f'r = {r:.3f}', xy=(0.05, 0.95), xycoords='axes fraction', fontsize=12)

plt.tight_layout()
plt.show()

## Summary

**What was done:**
- Transformed the long-format time series into daily aggregated summaries per patient
- Applied a sliding window approach (window = 7 days) to create instance-based features
- Engineered features per variable: mean, std, min, max, and trend (linear slope) over the window
- Added mood lag features (1, 2, 3 days back) and cyclical day-of-week encoding
- Target variable: next-day average mood

**Key observations:**
- The dataset is saved to `../data/dataset_features.csv` for use in Task 2 (modelling)
- Mood lag features (especially lag-1) are expected to be among the strongest predictors due to autocorrelation
- The sliding window captures both level (mean) and variability (std) of patient behaviour

**TODO:**
- Experiment with different window sizes (e.g., 3, 5, 14 days)
- Consider adding interaction features or ratio features
- Evaluate whether per-patient normalization improves model performance
- Test additional aggregation functions (e.g., median, skewness)
- Consider adding a patient-level identifier encoding for models that benefit from it